In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

adata = sc.read_h5ad("/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186.h5ad")
print(adata)

In [ ]:
# 标准化是必需的，确保不同细胞间可比
sc.pp.normalize_total(adata, target_sum=1e4)  # 标准化到每个细胞总计数为10000
sc.pp.log1p(adata)  # log转换


gene_list = [
'ITM2A', 'KLRB1', 'SPOCK2', 'ETS1', 'IL32', 'RORA', 'LCK', 'CD2', 'TNFRSF4', 'CD3E',
'CD69', 'CD247', 'TNFRSF18', 'CD96', 'CD7', 'SKAP1', 'LTB', 'GIMAP7', 'SYNE2', 'GATA3',
'TNFRSF25', 'PIK3IP1', 'IL7R', 'LBH', 'EVL', 'ITK', 'CD3D', 'ZAP70', 'ACAP1', 'LEF1',
'TUBA4A', 'PBXIP1', 'PYHIN1', 'PRKCH', 'FYN', 'SH2D1A', 'RHOH', 'CD27', 'LAT', 'GZMM',
'CLEC2D', 'TIGIT', 'ARID5B', 'ISG20', 'RASGRP1', 'TRAT1', 'CD40LG', 'CD3G', 'BTG1', 'CD6']


# 检查基因是否在数据中存在
genes_in_data = [gene for gene in gene_list if gene in adata.var_names]
genes_not_found = [gene for gene in gene_list if gene not in adata.var_names]

if genes_not_found:
    print(f"以下基因未在数据中找到: {genes_not_found}")
print(f"将使用 {len(genes_in_data)} 个基因进行打分")

# 方法1：使用score_genes（类似AddModuleScore）
# ctrl_size参数设置对照基因集大小（类似Seurat的nbin参数）
sc.tl.score_genes(
    adata, 
    gene_list=genes_in_data,
    score_name='module10_cell_score',  # 结果存储在adata.obs['T_cell_score']中
    ctrl_size=50,  # 对照基因集大小，默认为50
    use_raw=False  # 是否使用原始数据
)

# 查看结果
print(adata.obs['module10_cell_score'].describe())

# 可视化
sc.pl.umap(adata, color='module10_cell_score', cmap='RdBu_r')

In [ ]:
"""
TLS detection (two-stage size gates + overlap merge)
- 预筛：只看“核心合格点”的连通分量规模 MIN_CORE_CELLS
- 终筛：看“最终将要赋值的集合”的规模 MIN_FINAL_CELLS
- 合并1：按中心距离（旧逻辑）
- 合并2：按轮廓重叠（新逻辑；若 A/B 任意一方的已标记细胞落入对方轮廓 ≥ MIN_SHARED_CELLS_FOR_OVERLAP 即合并）
- 其它逻辑保持：动态分位阈值、B+T 组成校验、凸包/alpha-shape、QC 指标、可视化
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import scanpy as sc
from scipy.spatial import cKDTree, ConvexHull, Delaunay
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import entropy

# ================== 可调参数 ==================
# 基础参数
BASIS = "spatial"                  # 空间坐标所在的 obsm 键
SCORE_COL = "module10_cell_score"  # 打分列
CELL_TYPE_COL = "celltype_1_ZZM"   # 细胞类型列（如果有的话）

# 分数阈值策略（按样本动态分位数）
DYNAMIC_SCORE_THRESHOLD = True     # True: 每个样本用自身分布的分位数；False: 使用固定 SCORE_THRESHOLD
THRESHOLD_QUANTILE = 0.60          # 每个样本分数的分位数阈值（如需 Q75 改为 0.75）
SCORE_THRESHOLD = 0.20             # 当 DYNAMIC_SCORE_THRESHOLD=False 或样本分数全无效时的回退阈值

# TLS 识别参数
RADIUS = 40.0                      # 判邻域半径（μm；确保和坐标单位一致）
MIN_TOTAL_POS = 4                  # RADIUS 内分数>阈值的细胞总数阈值（含自身）；决定“核心合格点”是谁

# —— 两档规模阈值（仅此二者生效；不再使用 MIN_TLS_CELLS）——
MIN_CORE_CELLS  = 10               # 预筛：仅看“核心合格点”的最小规模（连通分量）
MIN_FINAL_CELLS = 10               # 终筛：看“最终将要赋值的集合”（凸包/alpha 内 & 未被占用）

# 形状识别参数
SHAPE_METHOD = "convex_hull"       # "convex_hull" 或 "alpha_shape"
ALPHA_VALUE = 100.0                # alpha shape 参数（当 <=0 时自动取 RADIUS）
FILL_ALL_CELLS_INSIDE = True       # True: 将形状内的所有细胞都标记为该 TLS（不只高分细胞）

# 分量处理顺序（避免“大圈吃小圈”的常见现象）
COMPONENT_ORDER = "small_first"    # "small_first" 或 "large_first"

# 合并参数
MERGE_NEARBY_TLS = True            # 按中心距离合并
MERGE_DISTANCE_THRESHOLD = 0    # 中心距离阈值（μm）

MERGE_OVERLAPPING_TLS = True       # ★ 新增：按轮廓重叠合并
MIN_SHARED_CELLS_FOR_OVERLAP = 1   # ★ 重叠判定：A 的已标记细胞落入 B 的轮廓（或反之）达到该数即合并
MAX_OVERLAP_MERGE_ROUNDS = 3       # 为了收敛，重叠合并最多迭代几轮

CHECK_BOUNDARY = True              # 是否检查边界效应
BOUNDARY_DISTANCE = 100.0          # 边界距离阈值（μm）

# 细胞类型验证参数 —— 必须同时包含 B 谱系任一 + T 谱系任一
VALIDATE_CELL_TYPES = True
MIN_CELL_TYPE_DIVERSITY = 2        # 至少两种类型（避免全是一类）
# B/T 谱系“任一即可”的匹配规则（不区分大小写）
REQUIRED_B_PATTERNS = [
    r"\bb\s*cells?",      # B cell / B cells
    # r"plasma"           # 如需允许浆细胞计入B谱系可解除注释
]
REQUIRED_T_PATTERNS = [
    r"\bt\s*cells?",      # T cell / T cells
    r"treg", r"cd4", r"cd8"
]

# 可视化参数
PLOT_COLS = 3
CELL_W, CELL_H = 12, 6
TLS_COLORS = [
    "#e41a1c","#1f78b4","#33a02c","#ff7f00","#6a3d9a","#e7298a",
    "#d95f02","#7570b3","#66a61e","#e6ab02","#1b9e77","#a6761d",
    "#fb9a99","#fdbf6f","#cab2d6","#b2df8a","#377eb8","#4daf4a",
    "#984ea3","#ffff33","#a65628","#f781bf","#fdae61","#fee08b",
    "#66c2a5","#fc8d62","#8da0cb","#e78ac3","#a6d854","#ffd92f",
    "#e5c494","#b3de69","#fccde5","#bc80bd","#ccebc5","#ffed6f",
]
NONE_COLOR = "#e0e0e0"

# 调试打印
VERBOSE = True
DEBUG_GATING = True          # 打印每个簇的门控细节（前若干个）
DEBUG_SHOW_FIRST_N = 30

# ================== 工具函数 ==================
def point_in_poly_mask(points, poly_xy):
    path = mpath.Path(poly_xy, closed=True)
    return path.contains_points(points)

def convex_hull_xy(xy):
    if xy.shape[0] < 3:
        return None
    try:
        hull = ConvexHull(xy)
        return xy[hull.vertices]
    except Exception:
        return None

def alpha_shape(points, alpha):
    """简化 alpha-shape：删去 Delaunay 的长边，再对剩余边端点取凸包（仍为凸，多用于弱抑制桥接）。"""
    if len(points) < 4:
        return convex_hull_xy(points)
    try:
        tri = Delaunay(points)
        edges = set()
        for simplex in tri.simplices:
            for i in range(3):
                for j in range(i+1, 3):
                    edges.add(tuple(sorted([simplex[i], simplex[j]])))
        edge_pts = []
        for i, j in edges:
            if np.linalg.norm(points[i] - points[j]) < alpha:
                edge_pts.extend([points[i], points[j]])
        if len(edge_pts) < 3:
            return convex_hull_xy(points)
        edge_pts = np.unique(np.array(edge_pts), axis=0)
        return convex_hull_xy(edge_pts)
    except Exception:
        return convex_hull_xy(points)

def effective_shape(points):
    """根据 SHAPE_METHOD / ALPHA_VALUE 计算多边形轮廓。"""
    if points.shape[0] < 3:
        return None
    if SHAPE_METHOD == "alpha_shape":
        alpha_eff = ALPHA_VALUE if (ALPHA_VALUE is not None and ALPHA_VALUE > 0) else max(RADIUS, 1.0)
        return alpha_shape(points, alpha_eff)
    else:
        return convex_hull_xy(points)

def calculate_cell_density(coords):
    if len(coords) < 3:
        return 0
    try:
        hull = ConvexHull(coords)
        return len(coords) / hull.volume  # 2D: volume=area
    except Exception:
        return 0

def calculate_compactness(coords):
    if len(coords) < 3:
        return 0
    try:
        hull = ConvexHull(coords)
        perim = 0.0
        for a, b in hull.simplices:
            perim += np.linalg.norm(coords[a] - coords[b])
        return (perim ** 2) / (4 * np.pi * hull.volume)
    except Exception:
        return 0

def shannon_entropy_from_series(s):
    if len(s) == 0:
        return 0.0
    counts = s.value_counts(dropna=True).values.astype(float)
    p = counts / counts.sum()
    return float(entropy(p))

def is_near_boundary(coords, all_coords, threshold=100):
    try:
        hull = ConvexHull(all_coords)
        boundary_points = all_coords[hull.vertices]
        tree = cKDTree(boundary_points)
        d, _ = tree.query(coords)
        return np.min(d) < threshold
    except Exception:
        return False

# ---- 细胞类型验证（B 任一 AND T 任一）----
def _match_any(series_str, patterns):
    if len(series_str) == 0:
        return False
    combined = "(?:" + ")|(?:".join(patterns) + ")"
    return series_str.str.contains(combined, regex=True, case=False, na=False).any()

def validate_tls_composition(ad, idx_final,
                             min_diversity=2,
                             b_patterns=REQUIRED_B_PATTERNS,
                             t_patterns=REQUIRED_T_PATTERNS):
    if CELL_TYPE_COL not in ad.obs.columns:
        return True  # 没有细胞类型信息则放行
    if len(idx_final) == 0:
        return False
    s = ad.obs.iloc[idx_final][CELL_TYPE_COL].astype(str)
    if s.nunique(dropna=True) < min_diversity:
        return False
    has_b = _match_any(s, b_patterns)
    has_t = _match_any(s, t_patterns)
    return bool(has_b and has_t)

def calculate_tls_metrics(ad, tls_id, coords_all):
    tls_mask = ad.obs["tls_degrow_id"] == tls_id
    tls_coords = coords_all[tls_mask]
    metrics = {
        "tls_id": tls_id,
        "n_cells": int(tls_mask.sum()),
        "mean_score": float(ad.obs.loc[tls_mask, SCORE_COL].mean()) if SCORE_COL in ad.obs else 0.0,
        "density": float(calculate_cell_density(tls_coords)),
        "compactness": float(calculate_compactness(tls_coords)),
        "is_boundary": bool(is_near_boundary(tls_coords, coords_all, BOUNDARY_DISTANCE)) if CHECK_BOUNDARY else False,
        "cell_type_diversity": int(ad.obs.loc[tls_mask, CELL_TYPE_COL].nunique()) if CELL_TYPE_COL in ad.obs.columns else 0,
        "cell_type_entropy": float(shannon_entropy_from_series(ad.obs.loc[tls_mask, CELL_TYPE_COL])) if CELL_TYPE_COL in ad.obs.columns else 0.0,
    }
    return metrics

# ---- 合并工具 ----
def merge_nearby_tls(ad, coords_all, distance_threshold=30.0):
    """按中心距离合并（就地修改 ad.obs['tls_degrow_id']）。"""
    tls_ids = np.array([x for x in ad.obs["tls_degrow_id"].unique() if x != "None"])
    if tls_ids.size < 2:
        return False
    centers = {}
    for tls_id in tls_ids:
        mask = (ad.obs["tls_degrow_id"] == tls_id).values
        centers[tls_id] = coords_all[mask].mean(axis=0)

    merged_any = False
    used = set()
    for i, t1 in enumerate(tls_ids):
        if t1 in used:
            continue
        group = [t1]; used.add(t1)
        for t2 in tls_ids[i+1:]:
            if t2 in used:
                continue
            if np.linalg.norm(centers[t1] - centers[t2]) < distance_threshold:
                group.append(t2); used.add(t2)
        if len(group) > 1:
            # 选“细胞数最多”的为主名
            sizes = [(g, int(np.sum(ad.obs["tls_degrow_id"].values == g))) for g in group]
            main = max(sizes, key=lambda x: x[1])[0]
            for t in group:
                if t == main: 
                    continue
                ad.obs.loc[ad.obs["tls_degrow_id"] == t, "tls_degrow_id"] = main
            merged_any = True
            if VERBOSE:
                print(f"   - 按中心距离合并TLS: {' + '.join(group)} -> {main}")
    return merged_any

def _tls_polygons(ad, coords_all):
    """返回 {tls_id: (poly_xy 或 None)}，用已标记细胞的坐标来估计轮廓。"""
    polys = {}
    for tls_id in ad.obs["tls_degrow_id"].unique():
        if tls_id == "None":
            continue
        mask = (ad.obs["tls_degrow_id"] == tls_id).values
        xy = coords_all[mask]
        poly = effective_shape(xy) if xy.shape[0] >= 3 else None
        polys[tls_id] = poly
    return polys

def merge_overlapping_tls(ad, coords_all, min_shared_cells=1):
    """
    若 A 与 B 的轮廓互相覆盖已标记细胞 >= min_shared_cells（A落入B 或 B落入A），则合并。
    采用并查集合并（一次收敛），并选择每个连通块细胞数最多者为主名。
    返回是否发生合并。
    """
    tls_ids = [x for x in ad.obs["tls_degrow_id"].unique() if x != "None"]
    if len(tls_ids) < 2:
        return False

    # 预计算：每个 TLS 的已标记细胞坐标 & 下标
    tls_masks = {}
    tls_coords = {}
    tls_indices = {}
    for t in tls_ids:
        m = (ad.obs["tls_degrow_id"] == t).values
        tls_masks[t] = m
        tls_coords[t] = coords_all[m]
        tls_indices[t] = np.where(m)[0]

    polys = _tls_polygons(ad, coords_all)

    # 并查集
    parent = {t: t for t in tls_ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # 建边：以“细胞落入对方轮廓”为重叠定义
    for i in range(len(tls_ids)):
        ti = tls_ids[i]
        poly_i = polys.get(ti, None)
        if poly_i is None:
            continue
        coords_i = tls_coords[ti]
        for j in range(i+1, len(tls_ids)):
            tj = tls_ids[j]
            poly_j = polys.get(tj, None)
            if poly_j is None:
                continue
            coords_j = tls_coords[tj]

            # A 的细胞是否落入 B 的轮廓
            inter_ij = int(np.sum(point_in_poly_mask(coords_i, poly_j)))
            # B 的细胞是否落入 A 的轮廓
            inter_ji = int(np.sum(point_in_poly_mask(coords_j, poly_i)))

            if (inter_ij >= min_shared_cells) or (inter_ji >= min_shared_cells):
                union(ti, tj)
                if VERBOSE:
                    print(f"   - 轮廓重叠判定：{ti} ↔ {tj}（A∈B:{inter_ij}, B∈A:{inter_ji}）→ 合并同组")

    # 根据并查集结果执行重命名
    groups = {}
    for t in tls_ids:
        r = find(t)
        groups.setdefault(r, []).append(t)

    merged_any = False
    for root, members in groups.items():
        if len(members) <= 1:
            continue
        # 选择每组中“细胞数最多”的作为主名
        sizes = [(m, int(np.sum(ad.obs["tls_degrow_id"].values == m))) for m in members]
        main = max(sizes, key=lambda x: x[1])[0]
        for m in members:
            if m == main:
                continue
            ad.obs.loc[ad.obs["tls_degrow_id"] == m, "tls_degrow_id"] = main
        merged_any = True
        if VERBOSE:
            print(f"   - 轮廓重叠合并TLS: {' + '.join(members)} -> {main}")

    return merged_any

# ================== 主流程 ==================
def detect_tls(adata):
    adata.obs["tls_degrow_id"] = "None"
    all_tls_metrics = []
    unique_samples = sorted(adata.obs["sample"].astype(str).unique())  # 固定顺序，保证复现
    tls_palette = {}
    tls_counter_global = 1

    for sample in unique_samples:
        print(f"\n>>> 处理样本: {sample}")
        msk = (adata.obs["sample"].astype(str) == sample)
        ad = adata[msk].copy()
        coords_all = ad.obsm[BASIS].astype(float)

        if SCORE_COL not in ad.obs.columns:
            print(f"   - 警告：未找到打分列 {SCORE_COL}，跳过")
            continue

        scores = ad.obs[SCORE_COL].values.astype(float)
        finite_scores = scores[np.isfinite(scores)]

        # —— 动态阈值：每个样本用自身分布的分位数；如无有效分数则回退到固定阈值
        if DYNAMIC_SCORE_THRESHOLD and finite_scores.size > 0:
            sample_thr = float(np.nanquantile(finite_scores, THRESHOLD_QUANTILE))
            if VERBOSE:
                print(f"   0) 动态阈值(Q{int(THRESHOLD_QUANTILE*100)}): {sample_thr:.4f}")
        else:
            sample_thr = SCORE_THRESHOLD
            if VERBOSE:
                print(f"   0) 使用固定阈值: {sample_thr:.4f}")

        # Step 1: 阈值筛分
        pos_mask = scores > sample_thr
        n_pos = int(pos_mask.sum())
        if VERBOSE:
            print(f"   1) 分数>{sample_thr:.4f}: {n_pos} 个")
        if n_pos == 0:
            continue

        pos_coords = coords_all[pos_mask]
        pos_index_in_ad = np.where(pos_mask)[0]

        # Step 2: 邻域计数（定义核心合格点）
        tree_pos = cKDTree(pos_coords)
        neigh_lists = tree_pos.query_ball_tree(tree_pos, r=RADIUS)  # 含自身
        total_counts = np.fromiter((len(L) for L in neigh_lists), dtype=int)
        eligible_mask_pos = total_counts >= max(MIN_TOTAL_POS, 1)   # 至少包含自身
        n_eligible = int(eligible_mask_pos.sum())
        if VERBOSE:
            print(f"   2) 满足邻域≥{MIN_TOTAL_POS}: {n_eligible} 个核心合格点")
        if n_eligible == 0:
            continue

        # Step 3: 合格细胞建图 & 连通分量
        elig_idx_pos = np.where(eligible_mask_pos)[0]
        elig_map = {p: i for i, p in enumerate(elig_idx_pos)}
        rows, cols = [], []
        for i_pos in elig_idx_pos:
            neigh = [j for j in neigh_lists[i_pos] if (j != i_pos) and eligible_mask_pos[j]]
            rows.extend([elig_map[i_pos]] * len(neigh))
            cols.extend([elig_map[j] for j in neigh])
        if not rows:
            print("   - 合格细胞之间没有连边，跳过。")
            continue
        n = n_eligible
        A = csr_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)), shape=(n, n))
        A = A.maximum(A.T)
        n_comp, labels = connected_components(A, directed=False)
        if VERBOSE:
            print(f"   3) 连通候选簇数: {n_comp}")

        # Step 4: 依规模排序（保护小团）
        comp_sizes = [(cid, int(np.sum(labels == cid))) for cid in range(n_comp)]
        reverse_flag = (COMPONENT_ORDER == "large_first")
        comp_sizes.sort(key=lambda x: x[1], reverse=reverse_flag)

        assigned = np.array(["None"] * ad.n_obs, dtype=object)
        valid_tls_count = 0

        # —— 门控统计 —— #
        gate_core_fail  = 0
        gate_final_fail = 0
        gate_bt_fail    = 0
        gate_pass       = 0
        printed = 0

        for comp_id, _size in comp_sizes:
            nodes = np.where(labels == comp_id)[0]
            core_size = nodes.size

            # 预筛：核心合格点规模
            if core_size < MIN_CORE_CELLS:
                gate_core_fail += 1
                if DEBUG_GATING and printed < DEBUG_SHOW_FIRST_N:
                    print(f"     [簇{comp_id:03d}] 核心={core_size} < MIN_CORE_CELLS={MIN_CORE_CELLS} → 预筛失败")
                    printed += 1
                continue

            # 该簇核心（在 ad 中的行号）
            pos_idx_in_pos = elig_idx_pos[nodes]
            ad_idx_core = pos_index_in_ad[pos_idx_in_pos]
            xy_core = coords_all[ad_idx_core]

            # 形状：决定“最终将要赋值”的集合（考虑未被占用的细胞）
            if FILL_ALL_CELLS_INSIDE and xy_core.shape[0] >= 3:
                poly = effective_shape(xy_core)
                if poly is not None:
                    inside = point_in_poly_mask(coords_all, poly)           # 所有细胞（含低分）
                    unassigned = (assigned == "None")
                    final_to_assign = np.where(inside & unassigned)[0]      # 最终准备赋值
                else:
                    ua = (assigned[ad_idx_core] == "None")
                    final_to_assign = ad_idx_core[ua]
            else:
                ua = (assigned[ad_idx_core] == "None")
                final_to_assign = ad_idx_core[ua]

            final_size = int(final_to_assign.size)

            # 终筛：最终圈内细胞规模
            if final_size < MIN_FINAL_CELLS:
                gate_final_fail += 1
                if DEBUG_GATING and printed < DEBUG_SHOW_FIRST_N:
                    print(f"     [簇{comp_id:03d}] 核心={core_size}，最终={final_size} < MIN_FINAL_CELLS={MIN_FINAL_CELLS} → 终筛失败")
                    printed += 1
                continue

            # 组成校验（可选）
            if VALIDATE_CELL_TYPES:
                ok = validate_tls_composition(ad, final_to_assign, min_diversity=MIN_CELL_TYPE_DIVERSITY)
                if not ok:
                    gate_bt_fail += 1
                    if DEBUG_GATING and printed < DEBUG_SHOW_FIRST_N:
                        print(f"     [簇{comp_id:03d}] 核心={core_size}，最终={final_size}，B+T 组成校验未过 → 失败")
                        printed += 1
                    continue

            # 通过全部条件，才创建/赋值 TLS
            tls_name = f"{sample}_TLS{tls_counter_global:03d}"
            tls_counter_global += 1
            valid_tls_count += 1
            gate_pass += 1

            assigned[final_to_assign] = tls_name

            # 颜色
            if tls_name not in tls_palette:
                cidx = (tls_counter_global - 2) % len(TLS_COLORS)
                tls_palette[tls_name] = TLS_COLORS[cidx]

            if DEBUG_GATING and printed < DEBUG_SHOW_FIRST_N:
                print(f"     [簇{comp_id:03d}] 核心={core_size}，最终={final_size} → 通过；赋名 {tls_name}")
                printed += 1

        # *** 同步到 ad，执行合并，再写回主 adata ***
        ad.obs["tls_degrow_id"] = assigned

        # 合并 1：按中心距离
        if MERGE_NEARBY_TLS:
            if VERBOSE:
                print("   5) 合并临近 TLS（中心距离） ...")
            merged = merge_nearby_tls(ad, coords_all, MERGE_DISTANCE_THRESHOLD)
            if VERBOSE and not merged:
                print("      - 无需按中心距离合并")

        # 合并 2：按轮廓重叠（迭代到收敛或达到上限轮数）
        if MERGE_OVERLAPPING_TLS:
            if VERBOSE:
                print("   6) 合并重叠 TLS（轮廓重叠） ...")
            round_id = 0
            while round_id < MAX_OVERLAP_MERGE_ROUNDS:
                round_id += 1
                merged = merge_overlapping_tls(ad, coords_all, min_shared_cells=MIN_SHARED_CELLS_FOR_OVERLAP)
                if not merged:
                    if VERBOSE:
                        print(f"      - 第 {round_id} 轮：无重叠可合并，停止")
                    break
                else:
                    if VERBOSE:
                        print(f"      - 第 {round_id} 轮：已合并，继续检查下一轮")

        # 写回主 adata
        adata.obs.loc[msk, "tls_degrow_id"] = ad.obs["tls_degrow_id"].values

        # 记录指标
        tls_in_sample = [x for x in ad.obs["tls_degrow_id"].unique() if x != "None"]
        for tls_id in tls_in_sample:
            m = calculate_tls_metrics(ad, tls_id, coords_all)
            m["sample"] = sample
            all_tls_metrics.append(m)

        if VERBOSE:
            n_assigned = int(np.sum(ad.obs["tls_degrow_id"].values != "None"))
            print(f"   7) 样本 {sample}: 标记 {n_assigned}/{ad.n_obs} 个细胞到 TLS；有效 TLS {len(tls_in_sample)} 个")
            # 门控统计总结
            total_checked = gate_core_fail + gate_final_fail + gate_bt_fail + gate_pass
            if total_checked > 0:
                print(f"      —— 门控统计 —— 预筛失败(core)<{MIN_CORE_CELLS}: {gate_core_fail}；"
                      f"终筛失败(final)<{MIN_FINAL_CELLS}: {gate_final_fail}；B+T失败: {gate_bt_fail}；通过: {gate_pass}")

    return tls_palette, all_tls_metrics

def print_tls_statistics(adata, tls_metrics):
    print("\n" + "="*60)
    print("TLS 统计信息")
    print("="*60)
    vc = adata.obs["tls_degrow_id"].value_counts()
    vc = vc[vc.index != "None"]
    print(f"总共识别出 {len(vc)} 个 TLS")
    if len(vc) > 0:
        print(f"平均每个 TLS 包含 {vc.mean():.1f} 个细胞")
        print(f"最大 TLS 包含 {vc.max()} 个细胞")
        print(f"最小 TLS 包含 {vc.min()} 个细胞")

    if tls_metrics:
        import pandas as pd
        df = pd.DataFrame(tls_metrics)
        print("\n前10个 TLS 的详细指标：")
        for _, row in df.head(10).iterrows():
            print(f"\n{row['tls_id']}:")
            print(f"  - 细胞数: {row['n_cells']}")
            print(f"  - 平均分数: {row['mean_score']:.3f}")
            print(f"  - 细胞密度: {row['density']:.2f}")
            print(f"  - 紧密度: {row['compactness']:.2f}")
            if 'cell_type_diversity' in row:
                print(f"  - 细胞类型多样性: {row['cell_type_diversity']}")
            if CHECK_BOUNDARY:
                print(f"  - 靠近边界: {'是' if row['is_boundary'] else '否'}")

        if len(df) > 10:
            print(f"\n... 还有 {len(df)-10} 个 TLS")
        print("\n汇总统计：")
        print(f"平均细胞密度: {df['density'].mean():.2f}")
        print(f"平均紧密度: {df['compactness'].mean():.2f}")
        if CHECK_BOUNDARY:
            nbd = int(df['is_boundary'].sum())
            print(f"靠近边界的 TLS: {nbd}/{len(df)} ({100*nbd/len(df):.1f}%)")

def visualize_tls(adata, tls_palette):
    samples = sorted(adata.obs["sample"].astype(str).unique())
    n_cols = PLOT_COLS
    n_rows = (len(samples) + n_cols - 1) // n_cols
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(CELL_W*n_cols, CELL_H*n_rows))
    axs = axs.flatten() if hasattr(axs, "flatten") else [axs]
    palette = {"None": NONE_COLOR}
    palette.update(tls_palette)

    for i, s in enumerate(samples):
        ax = axs[i]
        ad = adata[adata.obs["sample"].astype(str) == s].copy()
        sc.pl.embedding(
            ad, basis=BASIS, color="tls_degrow_id",
            frameon=False, size=30, show=False, ax=ax, palette=palette,
            title=f"{s}"
        )
        # 画轮廓
        for name in np.unique(ad.obs["tls_degrow_id"]):
            if name == "None":
                continue
            idx = (ad.obs["tls_degrow_id"] == name).values
            xy = ad.obsm[BASIS][idx]
            poly = effective_shape(xy)
            if poly is not None:
                color = palette.get(name, "black")
                ax.plot(np.r_[poly[:,0], poly[0,0]],
                        np.r_[poly[:,1], poly[0,1]],
                        color=color, lw=2.0, alpha=0.8)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")

    for j in range(len(samples), len(axs)):
        axs[j].axis("off")

    plt.suptitle(f"TLS Detection Results (Method: {SHAPE_METHOD})", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# ================== 执行 ==================
print("="*60)
print("开始 TLS 检测")
print("="*60)
if DYNAMIC_SCORE_THRESHOLD:
    print(f"分数阈值策略：按样本第 {int(THRESHOLD_QUANTILE*100)} 百分位动态设置")
else:
    print(f"分数阈值策略：固定阈值 = {SCORE_THRESHOLD}")
print(f"邻域={RADIUS}μm, 邻居≥{MIN_TOTAL_POS}")
print(f"规模门槛：核心≥{MIN_CORE_CELLS}（预筛），最终≥{MIN_FINAL_CELLS}（终筛）")
print(f"形状={SHAPE_METHOD}, alpha={ALPHA_VALUE}")
print(f"合并规则：中心距离≤{MERGE_DISTANCE_THRESHOLD}μm；轮廓重叠（>= {MIN_SHARED_CELLS_FOR_OVERLAP} 个已标记细胞互入）")
print(f"分量处理顺序：{COMPONENT_ORDER}")
if VALIDATE_CELL_TYPES:
    print(f"细胞类型验证：在最终集合上检查 B 任一 AND T 任一；最小多样性={MIN_CELL_TYPE_DIVERSITY}")

tls_palette, tls_metrics = detect_tls(adata)
print_tls_statistics(adata, tls_metrics)
visualize_tls(adata, tls_palette)

# 保存结果
adata.uns["tls_metrics"] = tls_metrics
adata.uns["tls_palette"] = tls_palette
print("\n完成：标签在 adata.obs['tls_degrow_id']，指标/配色保存在 adata.uns。")


In [ ]:
num_unique_samples = adata.obs['tls_degrow_id'].nunique()
print(f"唯一样本数量: {num_unique_samples}")

In [ ]:
# ==== 配置 ====
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

score_column = 'module10_cell_score'   # 改成你要看的列
OUTPUT_DIR = "/data/beifen/zhongmin/slide-tag/图/圈3级淋巴结构/2026-3-18/"  # 目标文件夹
os.makedirs(OUTPUT_DIR, exist_ok=True)
BASIS = globals().get("BASIS", "spatial")

_CANVAS_KW = dict(
    min_short_side=4.5,
    max_long_side=9.0,
    panel_width=1.55,
    panel_gap=0.18,
    left_pad=0.25,
    right_pad=0.18,
    top_pad=0.38,
    bottom_pad=0.22,
)

def safe_name(s: str) -> str:
    # 保留中英文、数字、下划线、点和短横，其它字符替换为下划线，避免非法文件名
    return re.sub(r'[^-\w\._\u4e00-\u9fff]+', '_', str(s))

def _adaptive_map_size_from_coords(coords_xy, min_short_side=4.5, max_long_side=9.0):
    if coords_xy is None or len(coords_xy) == 0:
        return (5.0, 7.0)
    x_span = float(np.ptp(coords_xy[:, 0])) if coords_xy.shape[1] >= 1 else 0.0
    y_span = float(np.ptp(coords_xy[:, 1])) if coords_xy.shape[1] >= 2 else 0.0
    x_span = max(x_span, 1.0)
    y_span = max(y_span, 1.0)

    if x_span >= y_span:
        map_w = max_long_side
        map_h = max(min_short_side, max_long_side * (y_span / x_span))
    else:
        map_h = max_long_side
        map_w = max(min_short_side, max_long_side * (x_span / y_span))

    return (map_w, map_h)

def _make_canvas_for_sample(coords_xy, **canvas_kw):
    map_w, map_h = _adaptive_map_size_from_coords(
        coords_xy,
        min_short_side=canvas_kw['min_short_side'],
        max_long_side=canvas_kw['max_long_side'],
    )
    fig_w = canvas_kw['left_pad'] + map_w + canvas_kw['panel_gap'] + canvas_kw['panel_width'] + canvas_kw['right_pad']
    fig_h = canvas_kw['bottom_pad'] + map_h + canvas_kw['top_pad']

    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_axes([
        canvas_kw['left_pad'] / fig_w,
        canvas_kw['bottom_pad'] / fig_h,
        map_w / fig_w,
        map_h / fig_h,
    ])
    panel_x = (canvas_kw['left_pad'] + map_w + canvas_kw['panel_gap']) / fig_w
    ax_panel = fig.add_axes([
        panel_x,
        canvas_kw['bottom_pad'] / fig_h,
        canvas_kw['panel_width'] / fig_w,
        map_h / fig_h,
    ])
    ax_panel.set_axis_off()
    return fig, ax, ax_panel

# 所有样本
unique_samples = sorted(adata.obs['sample'].astype(str).unique())

# 统一色阶：用全局分位数抑制极端值
vmin = adata.obs[score_column].quantile(0.01)
vmax = adata.obs[score_column].quantile(0.99)

# ==== 逐样本绘图并保存 ====
for sample_name in unique_samples:
    mask = adata.obs['sample'].astype(str).eq(sample_name).to_numpy()
    if not np.any(mask):
        continue
    sample_adata = adata[mask].copy()

    coords_xy = np.asarray(sample_adata.obsm[BASIS])[:, :2]
    fig, ax, ax_panel = _make_canvas_for_sample(coords_xy, **_CANVAS_KW)
    scores = sample_adata.obs[score_column].to_numpy(dtype=float)
    # 低分先画、高分后画，让高分 spot 尽量显示在上层
    draw_order = np.argsort(np.nan_to_num(scores, nan=-np.inf), kind='stable')
    coords_xy = coords_xy[draw_order]
    scores = scores[draw_order]
    scatter = ax.scatter(
        coords_xy[:, 0],
        coords_xy[:, 1],
        c=scores,
        s=30,
        cmap='RdBu_r',
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
        rasterized=True,
    )

    ax.set_title(f'{sample_name} (n={sample_adata.n_obs})', fontsize=36, pad=8)
    ax.set_aspect('equal', adjustable='box')
    ax.margins(0.02)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(''); ax.set_ylabel('')

    out_path = os.path.join(OUTPUT_DIR, f"{safe_name(sample_name)}_{score_column}.pdf")
    plt.savefig(out_path, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_path}")

# 共享色条：由于所有打分图都使用同一套全局 vmin/vmax，因此只需保存一次
shared_cbar_path = os.path.join(OUTPUT_DIR, f"{score_column}_shared_colorbar.pdf")
norm = Normalize(vmin=float(vmin), vmax=float(vmax))
sm = ScalarMappable(norm=norm, cmap='RdBu_r')
sm.set_array([])
fig_cb = plt.figure(figsize=(2.8, 6.2))
cax = fig_cb.add_axes([0.42, 0.12, 0.22, 0.76])
cbar = fig_cb.colorbar(sm, cax=cax)
cbar.ax.tick_params(labelsize=18, length=3, pad=3)
cbar.outline.set_visible(False)
fig_cb.text(0.5, 0.96, score_column, ha='center', va='top', fontsize=18)
plt.savefig(shared_cbar_path, dpi=300)
plt.close(fig_cb)
print(f"Shared colorbar saved: {shared_cbar_path}")

# （可选）导出每个样本的统计量
stats = (adata.obs[['sample', score_column]]
         .groupby('sample')[score_column]
         .agg(['count','mean','std','min','max'])
         .sort_index())
stats_path = os.path.join(OUTPUT_DIR, f"{score_column}_stats_by_sample.csv")
stats.to_csv(stats_path)
print(f"Stats saved: {stats_path}")

In [ ]:
# adata.write_h5ad('/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_11_06.h5ad')

In [ ]:
adata

In [ ]:
adata.write_h5ad('/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_11_10_4.h5ad')
adata

In [ ]:
# obs_data = adata.obs
# obs_data.to_csv("/data/beifen/zhongmin/slide-tag/H5AD转RDS/infercnv/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_11_10_4_2分之一obs.csv")

In [ ]:
中止

剔除3级淋巴结构中的恶性细胞和上皮细胞

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

In [ ]:
adata = sc.read_h5ad("/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_11_10_4.h5ad")
print(adata)

In [ ]:
# 设置 Pandas 的显示选项，确保所有内容都能打印出来
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，即显示所有行
pd.set_option('display.max_colwidth', None)  # 设置最大列宽为 None，即显示所有列内容

print(adata.obs['tls_degrow_id'].value_counts())

# 恢复默认的显示设置
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

In [ ]:
# 设置 Pandas 的显示选项，确保所有内容都能打印出来
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，即显示所有行
pd.set_option('display.max_colwidth', None)  # 设置最大列宽为 None，即显示所有列内容

print(adata.obs['celltype_3_ZZM'].value_counts())

# 恢复默认的显示设置
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

In [ ]:
import pandas as pd

# 1) 备份（可选）
adata.obs['tls_degrow_id_before_drop_epithelial'] = adata.obs['tls_degrow_id'].copy()

# 2) 只去掉 TLS 范围内的“上皮细胞”
types_to_strip = ['Malignant cells',"AT2","AT1","Multiciliated","SMC"]   # 如果以后要加别的类型，往这个列表里追加即可
mask = (
    adata.obs['celltype_1_ZZM'].isin(types_to_strip) &
    adata.obs['tls_degrow_id'].notna() &
    (adata.obs['tls_degrow_id'] != 'None')
)

print("将被移出 TLS 的细胞数：", int(mask.sum()))
# （可选）看一下这些细胞来自哪些 TLS
print(adata.obs.loc[mask, 'tls_degrow_id'].value_counts().head(10))

# 3) 设为 'None'（与你现有未圈定细胞保持一致）
adata.obs.loc[mask, 'tls_degrow_id'] = 'None'

# 4)（可选）快速核对：现在 TLS 内是否还残留上皮细胞
leftover = adata.obs[
    (adata.obs['celltype_1_ZZM'].isin(types_to_strip)) &
    adata.obs['tls_degrow_id'].notna() &
    (adata.obs['tls_degrow_id'] != 'None')
]
print("变更后仍在 TLS 标注内的上皮细胞数：", len(leftover))

In [ ]:
# 设置 Pandas 的显示选项，确保所有内容都能打印出来
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，即显示所有行
pd.set_option('display.max_colwidth', None)  # 设置最大列宽为 None，即显示所有列内容

print(adata.obs['tls_degrow_id'].value_counts())

# 恢复默认的显示设置
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

In [ ]:
adata

In [ ]:
# ========= 单样本逐张保存（仅最小改动：确保按调色表上色）=========
import os, re
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc

OUTPUT_DIR = "/data/beifen/zhongmin/slide-tag/图/圈3级淋巴结构"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 如外部未定义，给个默认
try:
    BASIS
except NameError:
    BASIS = "spatial"
try:
    NONE_COLOR
except NameError:
    NONE_COLOR = "#E6E8ED"

# 备用颜色轮（当调色表里没有该 TLS 名时，用稳定哈希取色）
TLS_COLORS = [
    "#e41a1c","#1f78b4","#33a02c","#ff7f00","#6a3d9a","#e7298a",
    "#d95f02","#7570b3","#66a61e","#e6ab02","#1b9e77","#a6761d"
]

def _safe_name(s: str) -> str:
    # 允许中英文、数字、下划线、点和短横，其它替换成下划线，避免文件名非法字符
    return re.sub(r'[^-\w\._\u4e00-\u9fff]+', '_', str(s))

def _lookup_color(name: str, by_name: dict, by_suffix: dict) -> str:
    # 1) 直接命中
    if name in by_name:
        return by_name[name]
    # 2) 用后缀命中：S22_TLS135 -> TLS135
    m = re.search(r"(TLS\d+)", str(name))
    if m and m.group(1) in by_suffix:
        return by_suffix[m.group(1)]
    # 3) 稳定哈希回退色
    return TLS_COLORS[hash(str(name)) % len(TLS_COLORS)]

def visualize_tls_save_each(adata, tls_palette, *, basis=BASIS, point_size=30, line_width=2.0):
    """
    按样本各画一张：按 tls_degrow_id 着色 + 叠加 TLS 轮廓，并保存 PDF。
    颜色来源优先级：
      ① 传入 tls_palette（dict）；
      ② adata.uns['tls_palette']（若存在）；
      ③ 名称后缀 TLS### 的映射；
      ④ 稳定哈希的备用色。
    """
    # 汇总全局调色：名称→颜色
    by_name = {"None": NONE_COLOR}
    if isinstance(tls_palette, dict) and len(tls_palette):
        by_name.update({str(k): v for k, v in tls_palette.items()})
    if "tls_palette" in adata.uns and isinstance(adata.uns["tls_palette"], dict):
        by_name.update({str(k): v for k, v in adata.uns["tls_palette"].items()})

    # 衍生：后缀 TLS### → 颜色
    by_suffix = {}
    for k, v in by_name.items():
        m = re.search(r"(TLS\d+)", k)
        if m:
            by_suffix.setdefault(m.group(1), v)

    samples = sorted(adata.obs["sample"].astype(str).unique())
    for s in samples:
        ad = adata[adata.obs["sample"].astype(str) == s].copy()
        if ad.n_obs == 0:
            continue

        # 当前样本的类别（含 None）
        cats_here = np.unique(ad.obs["tls_degrow_id"].astype(str))

        # 为当前样本补齐颜色字典（dict 形式，确保类别→颜色一一对应）
        local_palette = {}
        for name in cats_here:
            if name == "None":
                local_palette[name] = NONE_COLOR
            else:
                local_palette[name] = _lookup_color(name, by_name, by_suffix)

        # TLS 数（不计 None）
        n_tls = int(len([x for x in cats_here if x != "None"]))

        fig, ax = plt.subplots(figsize=(7, 6))
        sc.pl.embedding(
            ad,
            basis=basis,
            color="tls_degrow_id",
            frameon=False,
            size=point_size,          # 固定点大小
            show=False,
            ax=ax,
            palette=local_palette,    # ← dict: 类别→颜色（不再全灰）
            title=f"{s} | TLS: {n_tls}"
        )

        # 叠加每个 TLS 的轮廓（同色）
        for name in cats_here:
            if name == "None":
                continue
            idx = (ad.obs["tls_degrow_id"].astype(str) == name).values
            xy  = ad.obsm[basis][idx]
            # 你的凸包/alpha 形状函数；若未定义，这里仅在≥3点时画凸包
            poly = None
            if xy.shape[0] >= 3:
                try:
                    from scipy.spatial import ConvexHull
                    hull = ConvexHull(xy)
                    poly = xy[hull.vertices]
                except Exception:
                    poly = None
            if poly is not None:
                ax.plot(np.r_[poly[:, 0], poly[0, 0]],
                        np.r_[poly[:, 1], poly[0, 1]],
                        color=local_palette.get(name, "black"),
                        linewidth=line_width, alpha=0.8,
                        solid_joinstyle="round", solid_capstyle="round",
                        zorder=1)   # 让线在点的下层，减少视觉拥挤

        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(""); ax.set_ylabel("")

        out_path = os.path.join(OUTPUT_DIR, f"{_safe_name(s)}_TLS.pdf")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {out_path}")

# === 调用保持不变 ===
visualize_tls_save_each(
    adata, tls_palette,
    basis=BASIS,
    point_size=30,
    line_width=0.5
)

# 下面这一段与你原来一致（统计每样本每 TLS 的细胞数）
import pandas as pd, os
summary = (adata.obs
           .assign(sample=adata.obs["sample"].astype(str))
           .query("tls_degrow_id != 'None'")
           .groupby(["sample","tls_degrow_id"]).size()
           .rename("n_cells").reset_index())
summary_path = os.path.join(OUTPUT_DIR, "TLS_counts_per_sample.csv")
summary.to_csv(summary_path, index=False)
print(f"TLS counts saved: {summary_path}")


In [ ]:
adata.write_h5ad('/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_11_10_4.h5ad')
adata